In [1]:
from hydra import compose, initialize
from omegaconf import DictConfig, OmegaConf
import torch
import pytorch_lightning as pl
from retinalrisk.utils.callbacks import WritePredictionsDataFrame
from retinalrisk.training import setup_training

In [2]:
config_path = '/home/loockl/RetinalRisk/config/config.yaml'
model_path = '/sc-projects/sc-proj-ukb-cvd/results/models/retina/2af9tvdp/checkpoints/epoch=40-step=27962.ckpt'

In [3]:
args = OmegaConf.load(config_path)
print(OmegaConf.to_yaml(args))

defaults:
- setup: basic
- head: mlp
- datamodule: tte
- model: image
- training: default
- _self_
trainer:
  gpus:
  - 0
  precision: 16
  max_epochs: 10000
  num_sanity_val_steps: -1
  detect_anomaly: false
  amp_backend: native
  accumulate_grad_batches: 1



In [4]:
!pwd

/home/loockl/RetinalRisk/22_retina_phewas_evaluation


In [6]:
initialize(config_path="../config")
args = compose(config_name="config", overrides=[
    'setup.tags=[]',
    'datamodule.batch_size=32',
    'setup.use_data_artifact_if_available=False',
    'model.model_type=image',
    'model.encoder=simple_vit',
    '+model.encoder_image_size=1024',
    '+model.encoder_patch_size=32',
    '+model.encoder_num_classes=1000',
    'head=mlp',
    'head.kwargs.num_hidden=1024',
    'head.kwargs.num_layers=1',
    'head.dropout=0.5',
    'datamodule/covariates=no_covariates',
    'datamodule/augmentation=transformer_basic',
    'datamodule.img_size_to_gpu=1024',
    'datamodule.img_crop_ratio.train=[1.0]',
    'datamodule.img_crop_ratio.valid=1.0',
    'datamodule.img_crop_ratio.test=1.0'
])
#print(OmegaConf.to_yaml(args))

In [10]:
args.datamodule.label_definition.custom = '/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/220531/endpoints.csv'
datamodule, module, tags = setup_training(args)
args.model.restore_from_ckpt = model_path
cb = WritePredictionsDataFrame()

/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/phecode_definitions_220601.feather
/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/eids_211209.yaml
/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/baseline_covariates_220503.feather
/sc-projects/sc-proj-ukb-cvd/data/2_datasets_pre/211110_anewbeginning/artifacts/baseline_outcomes_220531.feather
Labels are...
['OMOP_4306655', 'phecode_001', 'phecode_002', 'phecode_002-1', 'phecode_003', 'phecode_004', 'phecode_004-1', 'phecode_004-2', 'phecode_004-3', 'phecode_005', 'phecode_005-1', 'phecode_005-2', 'phecode_007', 'phecode_007-1', 'phecode_008', 'phecode_009', 'phecode_010', 'phecode_011', 'phecode_012', 'phecode_015', 'phecode_015-2', 'phecode_019', 'phecode_020', 'phecode_020-1', 'phecode_024', 'phecode_025', 'phecode_030', 'phecode_050', 'phecode_050-4', 'phecode_052', 'phecode_052-1', 'phecode_052-3', 'phecode_052-31', 'phecode_052-32', 'phe

In [ ]:
cb.manual(args, datamodule, module)

Write predictions and patient embeddings
